# 2 · OIE extraction with Gemini  (per article)

Free-form open information extraction on one article at a time — **entities + types**: Gemini
invents the type label that best fits each entity, then defines each distinct type.
*(Relation / triple extraction is a later stage — not part of this notebook yet.)*

Set `ARTICLE_ID` for the single-article demo cell below, or list the ids you want in
`SELECTED_ARTICLES` and run the **batch** cell to process several articles in one run (defaults to
all). Output goes to `results/<ARTICLE_ID>/oie_entities.json`.

In [12]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

DATASET_DIR  = Path("dataset/articles")  # inputs:  <ARTICLE_ID>.txt
RESULTS_DIR  = Path("results")           # outputs: results/<ARTICLE_ID>/oie_*.json
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art2"                     # single article used by the section-1 / section-2 demo cells

# Which articles the BATCH cell processes, in one run.
#   None -> every dataset/articles/*.txt, sorted.
#   list -> exactly these ids, in this order, e.g. ["art1", "art2", "art20"].
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

_available = sorted(p.stem for p in DATASET_DIR.glob("*.txt"))
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no .txt in dataset):", missing)
    ARTICLE_IDS = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_IDS, "No runnable articles (need dataset/articles/<id>.txt). Check SELECTED_ARTICLES."
print("article (demo):", ARTICLE_ID, "| batch:", ARTICLE_IDS, "| results:", RESULTS_DIR)

article (demo): art2 | batch: ['art1', 'art2', 'art4', 'art5'] | results: results


In [9]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

Note: you may need to restart the kernel to use updated packages.


## 1. Entity + type extraction

In [15]:
# Free-form entity + type extraction -> results/<id>/oie_entities.json
OIE_ENTITY_SYSTEM_PROMPT = """You are an expert Entity Recognition and Typing system.
You are given a TEXT to analyze, and a REFERENCE_DATE (the article's publication date, ISO-8601) to anchor relative time expressions.
TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label, choosing the label freely based on what the entity is. You are not given a fixed set of types: invent whatever type label best describes each entity. The same surface text may be extracted more than once if, in different parts of the text, it clearly refers to different kinds of things (i.e. warrants different type labels). After extraction, produce a definition for every unique type label you used.
RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation in the "text" field. Do not invent, infer, or normalize the surface text of entities that are not explicitly mentioned. (Any resolved/normalized value goes in a separate field, never in "text" — see rule 7.)
2. Assign each entity a free-form type label of your own choosing. Pick the label that most precisely captures what kind of thing the entity is, given the evidence in the text. Do not restrict yourself to any predefined vocabulary.
3. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). Prefer a single clear noun or short noun phrase per label (e.g. "Person", "City", "Software Library", "Legal Case").
4. Choose the most specific label that still names a reusable category — one general enough to also apply to other entities of the same kind, not a label so narrow it fits essentially this entity alone. Favor a stable, commonly-recognized class over a hyper-specific or one-off label unlikely to recur across documents. Stay grounded in the text: do not over-generalize, and do not assert a specificity the evidence does not support.
5. Deduplicate on the (text, type) pair, not on text alone:
   - If the same surface text appears multiple times and refers to the same kind of thing each time, include it once.
   - If the same surface text refers to genuinely different kinds of things in different places (each supported by the text), include it once per distinct type, using the same surface text with the different type labels. Only do this when the evidence clearly supports more than one type for that exact mention text; do not split a single referent across types just because a label is uncertain.
6. "confidence" is your certainty in the assigned type for this mention, a value between 0 and 1. Set it per entity from how strongly the text supports the type — lower it when the mention is ambiguous, underspecified, or could reasonably take a different type. 
7. Temporal normalization — the "normalized_value" field:
   - For every entity that denotes a date or time — whether absolute ("24 June", "2026"), relative, or deictic ("Wednesday", "yesterday", "the last week of April", "this weekend", "next year") — resolve it to an ISO-8601 calendar value and put it in "normalized_value". Resolve relative/deictic expressions against REFERENCE_DATE, inferring direction (past vs. future) from the narrative context.
   - Use the granularity the text justifies: "YYYY-MM-DD" for a specific day, or a reduced form ("YYYY-MM", "YYYY") when only a coarser unit is warranted.
   - "normalized_value" holds a single point in time, or the start of a span. When the expression denotes a span (a weekend, a week, a month, a season, an explicit range, etc.), also set "normalized_end" to the end of the span (inclusive) at the justified granularity. For a single point in time, set "normalized_end" to null.
   - NEVER put the normalized value in "text": the original surface form always stays verbatim in "text".
   - For every non-temporal entity, and for any temporal expression you cannot confidently resolve, set both "normalized_value" and "normalized_end" to null.
8. After extracting all entities, collect every unique type label you used. For each, provide: (a) "definition": a concise one-sentence definition describing the category in general terms (what this kind of thing is); and (b) "synonyms": a short list of 2-5 alternative names or near-synonyms for the type (aka labels; e.g. for "Musician": "singer", "recording artist", "performer"). Both must be general and independent of the specific entities in the text. Do not include example instances, and do not include contrastive "distinct from" notes.
9. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once. The same (text, type) pair must not appear more than once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "normalized_value": "YYYY-MM-DD", "normalized_end": "YYYY-MM-DD or null", "confidence": <0-1> }
  ],
  "types": [
    { "type": "type_label", "definition": "A concise general definition of this type.", "synonyms": ["alternative name", "aka", "near-synonym"] }
  ]
}
Note: "normalized_value" and "normalized_end" are null for every non-temporal entity and for unresolvable time expressions; "normalized_end" is set only for spans (null for a single point in time)."""


import re

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline from the article header -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"


def extract_entities(article_id):
    """Run Gemini free-form entity+type OIE on one article -> results/<id>/oie_entities.json."""
    text = (DATASET_DIR / f"{article_id}.txt").read_text()
    ref_date = _reference_date(text)
    print(f"[{article_id}] ~{len(text)//4} tokens; REFERENCE_DATE={ref_date}; calling {GEMINI_MODEL} ...")
    user_message = (f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}" if ref_date else f"TEXT:\n{text}")
    oie = gemini_json(OIE_ENTITY_SYSTEM_PROMPT, user_message)
    out_dir = RESULTS_DIR / article_id
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "oie_entities.json").write_text(json.dumps(oie, indent=2, ensure_ascii=False))
    n_norm = sum(1 for e in oie.get("entities", []) if e.get("normalized_value"))
    print(f"  latency {last_latency_s:.2f}s | {len(oie.get('entities', []))} entities "
          f"({n_norm} time-normalized), {len(oie.get('types', []))} distinct types -> {out_dir / 'oie_entities.json'}")
    return oie

# oie = extract_entities(ARTICLE_ID)

## 3. Batch — process the selected articles (`ARTICLE_IDS`)

In [16]:
# Runs entity extraction for each article in ARTICLE_IDS (choose them via SELECTED_ARTICLES in the
# config cell). Re-runs overwrite existing outputs. (Relation/triple extraction comes in a later stage.)
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    extract_entities(aid)
print(f"\ndone — extracted entities for {len(ARTICLE_IDS)} article(s)")

============================================================ art1
[art1] ~1170 tokens; REFERENCE_DATE=2026-07-03; calling gemini-2.5-flash ...
  latency 121.64s | 107 entities (22 time-normalized), 51 distinct types -> results/art1/oie_entities.json
============================================================ art2
[art2] ~1865 tokens; REFERENCE_DATE=2026-07-03; calling gemini-2.5-flash ...
  latency 70.70s | 97 entities (15 time-normalized), 48 distinct types -> results/art2/oie_entities.json
============================================================ art4
[art4] ~666 tokens; REFERENCE_DATE=2025-09-03; calling gemini-2.5-flash ...
  latency 103.86s | 77 entities (7 time-normalized), 24 distinct types -> results/art4/oie_entities.json
============================================================ art5
[art5] ~1147 tokens; REFERENCE_DATE=2026-07-03; calling gemini-2.5-flash ...
  latency 57.56s | 92 entities (6 time-normalized), 45 distinct types -> results/art5/oie_entities.json

done — 